In [ ]:
# pip install torch torchvision timm sentence-transformers pandas pyarrow

import torch
from torch.utils.data import Dataset, DataLoader, IterableDataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import pyarrow.parquet as pq  # если данные в parquet — в 10 раз быстрее и меньше памяти
from sentence_transformers import SentenceTransformer
import timm
import numpy as np
from pathlib import Path

# ------------------- 1. Чтение большого CSV/Parquet по чанкам -------------------
class BigDataIterable(IterableDataset):
    def __init__(self, file_path, text_cols=['title', 'description'], image_col='image_path', label_col='label'):
        self.file_path = file_path
        self.text_cols = text_cols
        self.image_col = image_col
        self.label_col = label_col
        
        # Если parquet — супербыстро и мало памяти
        if file_path.endswith('.parquet'):
            self.table = pq.ParquetFile(file_path)
            self.num_rows = self.table.metadata.num_rows
        else:
            # для csv тоже можно по чанкам
            pass

    def __iter__(self):
        # Итерируемся по батчам прямо из файла
        for batch in self.table.iter_batches(batch_size=1024, columns=self.text_cols + [self.image_col, self.label_col]):
            df_batch = batch.to_pandas()
            for _, row in df_batch.iterrows():
                yield {
                    'text': ' '.join(str(row[col]) for col in self.text_cols if pd.notna(row[col])),
                    'image_path': row[self.image_col],
                    'label': row[self.label_col] if self.label_col in row else -1
                }

# ------------------- 2. Dataset с ленивой загрузкой изображений -------------------
class LazyImageTextDataset(IterableDataset):
    def __init__(self, data_iterable, img_size=384, st_model_name="intfloat/multilingual-e5-small"):
        self.data_iterable = data_iterable
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        # Модель для текста — грузим один раз
        self.st_model = SentenceTransformer(st_model_name, device='cuda' if torch.cuda.is_available() else 'cpu')
        
    def __iter__(self):
        for item in self.data_iterable:
            try:
                # Ленивая загрузка изображения только в момент __getitem__
                img = Image.open(item['image_path']).convert('RGB')
                img = self.transform(img)
                
                # Текст → эмбеддинг (можно батчить, но для простоты по одному)
                text_emb = self.st_model.encode(item['text'], normalize_embeddings=True, show_progress_bar=False)
                text_emb = torch.tensor(text_emb, dtype=torch.float32)
                
                label = torch.tensor(item['label'], dtype=torch.long) if item['label'] != -1 else None
                
                yield img, text_emb, label
            except Exception as e:
                # Важно: пропускаем битые картинки!
                print(f"Ошибка загрузки {item['image_path']}: {e}")
                continue

# ------------------- Использование -------------------
train_dataset = LazyImageTextDataset(
    BigDataIterable('train_10M_rows.parquet'),
    img_size=384
)

train_loader = DataLoader(train_dataset, batch_size=64, num_workers=8, pin_memory=True)

In [ ]:
class MultimodalModel(torch.nn.Module):
    def __init__(self, num_classes, img_backbone='eva02_large_patch14_448', text_dim=384):
        super().__init__()
        self.img_model = timm.create_model(img_backbone, pretrained=True, num_classes=0)  # без головы
        self.text_proj = torch.nn.Linear(text_dim, 512)
        self.classifier = torch.nn.Sequential(
            torch.nn.LayerNorm(2048 + 512),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(2048 + 512, num_classes)
        )
    
    def forward(self, img, text_emb):
        img_feat = self.img_model(img)           # (B, 2048)
        text_feat = self.text_proj(text_emb)     # (B, 512)
        fused = torch.cat([img_feat, text_feat], dim=1)
        return self.classifier(fused)

model = MultimodalModel(num_classes=10).cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
criterion = torch.nn.CrossEntropyLoss()

# Обучение на потоковых данных (может идти сутками — память не растёт!)
model.train()
for epoch in range(10):
    for imgs, text_embs, labels in train_loader:
        imgs = imgs.cuda()
        text_embs = text_embs.cuda()
        labels = labels.cuda()
        
        preds = model(imgs, text_embs)
        loss = criterion(preds, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
# Шаг 1: предпосчитываем эмбеддинги изображений и текста по частям
def save_embeddings_in_chunks(parquet_path, chunk_size=100_000):
    model_img = timm.create_model('fastvit_ma36', pretrained=True, num_classes=0).cuda().eval()
    model_text = SentenceTransformer("intfloat/multilingual-e5-small").cuda()
    
    for i, batch in enumerate(pq.ParquetFile(parquet_path).iter_batches(batch_size=chunk_size)):
        df = batch.to_pandas()
        
        # изображения
        img_feats = []
        for path in df['image_path']:
            img = Image.open(path).convert('RGB')
            img = transform(img).unsqueeze(0).cuda()
            with torch.no_grad():
                feat = model_img(img).cpu().numpy()
            img_feats.append(feat)
        
        # текст
        text_feats = model_text.encode(df['text'].tolist(), batch_size=512, normalize_embeddings=True)
        
        # сохраняем чанк
        np.savez(f'embeddings_chunk_{i}.npz', 
                 img=np.vstack(img_feats), 
                 text=text_feats, 
                 labels=df['label'].values if 'label' in df else None)

In [ ]:
# Используем dataset из папки + ImageFolder + добавляем текст через отдельный csv
from torch.utils.data import Dataset
import os

class ImageTextDataset(Dataset):
    def __init__(self, img_dir, csv_path, transform=None):
        self.img_dir = Path(img_dir)
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.img_paths = [self.img_dir / fname for fname in os.listdir(img_dir)]
    
    def __len__(self): return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform: img = self.transform(img)
        
        row = self.df.iloc[idx]  # или по имени файла сопоставляем
        text_emb = torch.load(f"text_embeddings/{img_path.stem}.pt")  # предпосчитанные
        
        return img, text_emb, torch.tensor(row['label'])